# Cleartext DP training
This notebook runs the differentially private (DP) training for the SoK experiments without cryptographic protections in a centralized setting.

In [ ]:
import torch
import math
from time import time
import matplotlib.pyplot as plt
import numpy as np
import sys
import crypten
import torch.nn as nn
from torch.nn.functional import cross_entropy

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

from opacus.data_loader import DPDataLoader as PoissonDataLoader
from opacus import PrivacyEngine

import nest_asyncio
nest_asyncio.apply()

import os
from pfl.internal.ops import pytorch_ops
from pfl.internal.ops.selector import set_framework_module

device = "cuda" if torch.cuda.is_available() else "cpu"

# If on Apple Silicon, use MPS
# device = "mps"
# os.environ['PFL_PYTORCH_DEVICE'] = device
set_framework_module(pytorch_ops)


from pfl.privacy import GaussianMechanism, PLDPrivacyAccountant

sys.path.append('./dataset/')
from dataset.fashion_mnist.load_preprocess import load_and_preprocess_fashion_mnist
from dataset.mnist.load_preprocess import load_and_preprocess_mnist

sys.path.append('./utils/')
from utils.models import ThreeLayerNN
from utils.mpc_dpsgd_trainer import DP_Trainer

## Model
The model used for the experiments is a 3 layers neural network

In [ ]:
model = ThreeLayerNN(
    input_size=784,
	hidden_size=100,
	output_size=10
)

## Dataset
The experiments can be run on either MNIST or Fashion-MNIST datasets by changing the `dataset` variable below.

In [ ]:
dataset_name = "mnist" 
# dataset = "fashion_mnist"

In [ ]:
torch.random.manual_seed(0)
np.random.seed(0)

crypten.init()

if dataset_name == "mnist":
	train_data, val_data = load_and_preprocess_mnist(scaling=True, normalization=True)
elif dataset_name == "fashion_mnist":
	train_data, val_data = load_and_preprocess_fashion_mnist(scaling=True, normalization=True)
else:
	raise ValueError(f"Unsupported dataset name: {dataset_name}")

x_train, y_train = train_data.data, train_data.targets
x_val, y_val = val_data.data, val_data.targets

y_train = y_train.squeeze()
y_val = y_val.squeeze()


print(f"Max value of x_train: {x_train.max()}, Min value of x_train: {x_train.min()}")
print(x_train.shape, y_train.shape)
print(x_val.shape, y_val.shape)

## DP Accountant Setup

We use the Privacy Loss Distribution (PLD) accountant for tracking the privacy loss during training. Here, we set up the DP accountant and the Gaussian mechanism used for adding noise to the gradients.

In [ ]:
dataset_size = x_train.shape[0]
batch_size = 500
epsilon = 8.0
delta = 10e-5
epochs = 10
sampling_probability = batch_size/dataset_size
clipping_bound = 4.0
learning_rate = 0.1

pld_accountant = PLDPrivacyAccountant(
    num_compositions=epochs*(math.ceil(dataset_size / batch_size)),
    sampling_probability=sampling_probability,
    mechanism='gaussian',
    epsilon=epsilon,
    delta=delta
)

pld_gaussian = GaussianMechanism.from_privacy_accountant(
    accountant=pld_accountant, clipping_bound=clipping_bound)

noise_multiplier = pld_gaussian._relative_noise_stddev
print(f"noise_multiplier: {noise_multiplier}")

## DP training 

For DP training, we tested two variants, i.e., one using the DP library Opacus and the other one using our custom Trainer.
The main difference between the two approaches is that Opacus supports optimize per-example clipping while our Trainer not. We implement a naive version of DP training (i.e., with non-optimized clipping) to compare the performance of the two variants.

In [1]:
def validate(model, val_loader, criterion):
    model.eval()  # Set the model to evaluation mode
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation for validation
        for data, target in val_loader:
            output = model(data)
            val_loss += criterion(output, target).item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)

    val_loss /= len(val_loader)
    accuracy = 100.0 * correct / total
    return val_loss, accuracy

### Training with Opacus

In [ ]:
batch_size = 500
train_dataset = TensorDataset(torch.tensor(x_train), torch.tensor(y_train))
val_dataset = TensorDataset(torch.tensor(x_val), torch.tensor(y_val))
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

privacy_engine = PrivacyEngine(
    accountant="prv"
)

model, optimizer, train_dataloader = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=train_dataloader, # Already a PoissonDataLoader
    noise_multiplier=noise_multiplier,
    max_grad_norm=clipping_bound,
    poisson_sampling=False 
)

# now execute the training loop with validation every epoch using torch
criterion = torch.nn.CrossEntropyLoss()


In [ ]:
torch.manual_seed(0)
np.random.seed(0)

for epoch in range(1, 11):  # 10 epochs
	model.train()  # Set the model to training mode
	for batch_idx, (data, target) in enumerate(train_dataloader):
		optimizer.zero_grad()
		output = model(data)
		loss = criterion(output, target)
		loss.backward()
		optimizer.step()
        
		epsilon = privacy_engine.get_epsilon(delta=10e-5)
		print(f'Epoch: {epoch} \t Loss: {loss.item()} \t (ε = {epsilon:.2f}, δ = 1e-5)')

    # Perform validation
	val_loss, val_accuracy = validate(model, val_dataloader, criterion)
	epsilon = privacy_engine.get_epsilon(delta=10e-5)
	print(f'Epoch: {epoch} \t '
		f'Training Loss: {loss.item():.6f} \t '
		f'Validation Loss: {val_loss:.6f} \t '
		f'Validation Accuracy: {val_accuracy:.2f}% \t '
		f'(ε = {epsilon:.2f}, δ = 1e-5)')

### DP-Training with custom Trainer 

In [ ]:
device ="mps"
dp_trainer = DP_Trainer(
    model=model,
	noise_multiplier=noise_multiplier,
    noise_type="central",
	batch_size=batch_size,
    lr = learning_rate,
	num_epochs=epochs,
    verbose = False,
	device=device,
    subsampling_type='poisson',
    sampling_rate=batch_size/dataset_size,
    num_labels=10,
    clipping_threshold=clipping_bound
)

In [ ]:
torch.random.manual_seed(0)
np.random.seed(0)

dp_trainer.train_and_validate(
	x = x_train, 
	y = y_train,
	x_val = x_val,
	y_val = y_val,
	validation_freq=1
)